In [1]:
import os
import shutil
import datetime
#os.environ["KAGGLE_API_TOKEN"] = "KGAT_898cde7db386649d29b771373549581f" 
import opendatasets as od
import pandas as pd
import string
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from ModelArchitecture.StandardCharRNN import CharRNNClassic

import matplotlib.pyplot as plt
from tqdm import tqdm
CHAR_LEN = 256
allowed_chars = set(
    string.ascii_letters +   
    string.digits +          
    string.punctuation +     
    " \n\t"                 
)

save_dir = "/Users/em/Downloads/COGS 185/FinalProj/weight_files"


In [2]:
#Make a function that finds which plot summaries mention authors, then filter out all in dataset that does this
def mentions_author(row):
    author = str(row["Author"]).lower()
    summary = str(row["Plot_Sum"]).lower()
    
    names = author.split()
    
    first= names[0]
    last = names[-1]
    
    return first in summary or last in summary

def mentions_title(row):
    title = str(row["Book_Title"]).lower()
    summary = str(row["Plot_Sum"]).lower()
    
    
    return title in summary



In [3]:

def seq_to_indices(seq, char_to_idx):
    return torch.tensor([char_to_idx[c] for c in seq])


def make_input_target(seq, char_to_idx):

    indices = seq_to_indices(seq, char_to_idx)

    input_idx  = indices[:-1]
    target_idx = indices[1:]


    return input_idx, target_idx

class CharDataset(Dataset):

    def __init__(self, windows, char_to_idx):
        self.windows = windows
        self.char_to_idx = char_to_idx

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):

        seq = self.windows[idx]

        x, y = make_input_target(seq, self.char_to_idx)
        return x, y
    



In [4]:
od.download('https://www.kaggle.com/datasets/ymaricar/cmu-book-summary-dataset')

Skipping, found downloaded files in "./cmu-book-summary-dataset" (use force=True to force download)


In [5]:
columns = ["WikiID", "FreebaseID", "Book_Title","Author","Pub_date", "Book_Genre","Plot_Sum"]
book_summary_dataset = pd.read_csv("./cmu-book-summary-dataset/booksummaries.txt", sep="\t", header=None, names = columns)
print(f"There are {book_summary_dataset.shape[0]} book summaries in the dataset initially")


There are 16559 book summaries in the dataset initially


In [6]:
pure_story_summary = book_summary_dataset[~book_summary_dataset.apply(mentions_author, axis=1)]
pure_story_summary = pure_story_summary[~pure_story_summary.apply(mentions_title, axis=1)]
def is_valid(text):
    return all(c in allowed_chars for c in text)
pure_story_summary = pure_story_summary[pure_story_summary["Plot_Sum"].apply(is_valid)]
pure_story_summary = pure_story_summary[~pure_story_summary["Plot_Sum"].str.contains(r"[<>]", regex=True, na=False)]

pure_story_summary["Plot_Sum"] = pure_story_summary["Plot_Sum"].str.strip()
pure_story_summary["Plot_Sum"] = "<" + pure_story_summary["Plot_Sum"] + ">"
pure_story_summary = pure_story_summary[pure_story_summary['Plot_Sum'].str.len() >= CHAR_LEN]
pure_story_summary["Sum_Char_Count"] = pure_story_summary["Plot_Sum"].str.len()
single_string = pure_story_summary['Plot_Sum'].str.cat(sep='')
chars = sorted(set(single_string))
print(len(chars))
print(f"After filtering out all summaries that include the authors name we have {pure_story_summary.shape[0]} book summaries in the dataset")


94
After filtering out all summaries that include the authors name we have 6220 book summaries in the dataset


In [7]:
pure_story_summary.head()

,WikiID,FreebaseID,Book_Title,Author,Pub_date,Book_Genre,Plot_Sum,Sum_Char_Count
6,2890,/m/011zx,A Wizard of Earthsea,Ursula K. Le Guin,1968,"{""/m/0dwly"": ""Children's literature"", ""/m/01hm...","<Ged is a young boy on Gont, one of the larger...",5851
8,4081,/m/01b4w,Blade Runner 3: Replicant Night,K. W. Jeter,1996-10-01,"{""/m/06n90"": ""Science Fiction"", ""/m/014dfn"": ""...","<Living on Mars, Deckard is acting as a consul...",320
9,4082,/m/01b56,Blade Runner 2: The Edge of Human,K. W. Jeter,1995-10-01,"{""/m/06n90"": ""Science Fiction"", ""/m/014dfn"": ""...",<Beginning several months after the events in ...,1787
17,4451,/m/01f9l,Book of Jonah,NaN,NaN,NaN,<The plot centers on a conflict between Jonah ...,1760
21,6628,/m/01y92,Children of Dune,Frank Herbert,1976,"{""/m/06n90"": ""Science Fiction"", ""/m/014dfn"": ""...",<Nine years after Emperor Paul Muad'dib walked...,3827


In [8]:
#Print average character amount to understand breadth of dataset
total_char = 0
avg_char = 0

total_char = pure_story_summary["Sum_Char_Count"].sum()

avg_char = total_char / pure_story_summary.shape[0]
print(f"Total characters is {total_char}")
print(f"Average characters is {round(avg_char)}")


Total characters is 11609470
Average characters is 1866


In [9]:
#Creating the windowed dataset for the RNN
windowed_list = []
window_size = 256
stride_size = 64
for sample_num in tqdm(range(pure_story_summary.shape[0])):
    i = 0
    window_sample =torch.tensor([])
    current_sample = pure_story_summary.iloc[sample_num]["Plot_Sum"]
    sample_done = False
    
    while sample_done is False:
        if((i*stride_size + window_size) < len(current_sample)):
            window_sample = current_sample[i*stride_size:(i*stride_size + window_size)]
            #window_sample = current_sample[(len(current_sample) - window_size):]
            if(len(window_sample) != window_size):
                print("For full sample",len(window_sample))
        else:
            
            window_sample = current_sample[(len(current_sample) - window_size):]
            if(len(window_sample) != window_size):
                print("For leftover sample",len(window_sample))
            sample_done = True
        windowed_list.append(window_sample)
        i+=1
    print(f"{i} windowed samples made for this sample")




 32%|███▏      | 1967/6220 [00:00<00:00, 19663.08it/s]

89 windowed samples made for this sample
2 windowed samples made for this sample
25 windowed samples made for this sample
25 windowed samples made for this sample
57 windowed samples made for this sample
97 windowed samples made for this sample
47 windowed samples made for this sample
125 windowed samples made for this sample
83 windowed samples made for this sample
170 windowed samples made for this sample
15 windowed samples made for this sample
21 windowed samples made for this sample
52 windowed samples made for this sample
52 windowed samples made for this sample
100 windowed samples made for this sample
85 windowed samples made for this sample
43 windowed samples made for this sample
36 windowed samples made for this sample
71 windowed samples made for this sample
99 windowed samples made for this sample
56 windowed samples made for this sample
12 windowed samples made for this sample
89 windowed samples made for this sample
18 windowed samples made for this sample
24 windowed sa

 66%|██████▌   | 4076/6220 [00:00<00:00, 20501.70it/s]

37 windowed samples made for this sample
4 windowed samples made for this sample
20 windowed samples made for this sample
20 windowed samples made for this sample
12 windowed samples made for this sample
15 windowed samples made for this sample
3 windowed samples made for this sample
2 windowed samples made for this sample
14 windowed samples made for this sample
17 windowed samples made for this sample
17 windowed samples made for this sample
20 windowed samples made for this sample
10 windowed samples made for this sample
11 windowed samples made for this sample
15 windowed samples made for this sample
4 windowed samples made for this sample
11 windowed samples made for this sample
3 windowed samples made for this sample
5 windowed samples made for this sample
2 windowed samples made for this sample
13 windowed samples made for this sample
25 windowed samples made for this sample
8 windowed samples made for this sample
41 windowed samples made for this sample
53 windowed samples made

100%|██████████| 6220/6220 [00:00<00:00, 20613.78it/s]

6 windowed samples made for this sample
20 windowed samples made for this sample
3 windowed samples made for this sample
6 windowed samples made for this sample
45 windowed samples made for this sample
17 windowed samples made for this sample
39 windowed samples made for this sample
8 windowed samples made for this sample
3 windowed samples made for this sample
5 windowed samples made for this sample
2 windowed samples made for this sample
24 windowed samples made for this sample
3 windowed samples made for this sample
11 windowed samples made for this sample
7 windowed samples made for this sample
6 windowed samples made for this sample
19 windowed samples made for this sample
6 windowed samples made for this sample
35 windowed samples made for this sample
253 windowed samples made for this sample
51 windowed samples made for this sample
51 windowed samples made for this sample
11 windowed samples made for this sample
15 windowed samples made for this sample
5 windowed samples made fo

In [10]:
all_text = "".join(pure_story_summary["Plot_Sum"])

chars = sorted(list(set(all_text)))
vocab_size = len(chars)

char_to_idx = {c:i for i,c in enumerate(chars)}
idx_to_char = {i:c for i,c in enumerate(chars)}
dataset = CharDataset(windowed_list, char_to_idx)


window_sample_loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True
)




In [11]:
def generate_text(model, char_to_idx, idx_to_char, max_length=2000, start_text = "<", stop_char = ">", temperature=0.8, device="cpu"):
    
    model.eval()
    
    input_seq = torch.tensor([[char_to_idx[c] for c in start_text]], device=device)
    hidden = model.init_hidden(1)
    
    if isinstance(hidden, tuple):
        hidden = (hidden[0].to(device), hidden[1].to(device))
    else:
        hidden = hidden.to(device)

    generated = start_text

    # Prime the model with the seed text
    for i in range(len(start_text) - 1):
        _, hidden = model.forward2(input_seq[:, i], hidden)

    current_char = input_seq[:, -1]

    for _ in range(max_length):

        output, hidden = model.forward2(current_char, hidden)

        logits = output.squeeze() / temperature
        probs = torch.softmax(logits, dim=0)

        next_idx = torch.multinomial(probs, 1).item()
        next_char = idx_to_char[next_idx]

        generated += next_char
        if(next_char == stop_char):
            break;
        current_char = torch.tensor([next_idx], device=device)


    return generated

In [13]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device) 
loss_func = nn.CrossEntropyLoss() 

hidden_size = 512
layer_amount = 2
embedding_size = 128
epochs = 20

net = CharRNNClassic(input_size = vocab_size, hidden_size = 512,output_size = vocab_size, embedding_size= embedding_size, n_layers = layer_amount)
net = net.to(device)

opt = torch.optim.Adam(net.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5)

total_loss = 0
loss = 0
best_loss = torch.inf
best_epoch = -1
batches = len(window_sample_loader)
print("Dataloader has",batches, "batches")

if(os.path.isdir(save_dir) is False):
    os.makedirs(save_dir)

timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

early_close = False
for epoch in range(epochs):
    net.train()
    if(early_close):
        batch_idx = 0
    for batch in tqdm(window_sample_loader, desc = f"Progress for Epoch {epoch}"):
        x,y = batch
        x = x.to(device)
        y = y.to(device)
        batch_size = x.shape[0]
        
        opt.zero_grad()
        hidden = net.init_hidden(batch_size)
        hidden = hidden.to(device)

        output, hidden = net(x, hidden)
        #print("Output Shape", output.shape)
        #print("Target Shape", y.shape)
        output = output.reshape(-1, vocab_size)
        target = y.reshape(-1)

        loss = loss_func(output, target)
        loss.backward()             # Backward. 
        opt.step()
        total_loss += loss.item()
        if(early_close):
            batch_idx += 1
            if(batch_idx == 10):
                print("Closed early")
                break;
    scheduler.step()
    if(early_close):
        avg_loss = total_loss / 10
    else:
        avg_loss = total_loss / batches

    print(f"In epoch {epoch+1}, the average loss is {avg_loss}")
    
    
    if(avg_loss < best_loss):
        print(f"New best loss, printing new model weights file and shopwing generated text for epoch {epoch}")
        generated_text = generate_text(net, char_to_idx=char_to_idx, idx_to_char=idx_to_char, device=device)
        print(f"Generated Text:\n{generated_text}")
        if(early_close):
            file_name = f"classic_rnn_epoch_eclose_{epoch}_{timestamp}.pt"
        else:
            file_name = f"classic_rnn_epoch_{epoch}_{timestamp}.pt"
        weights_path = os.path.join(save_dir, file_name)
        torch.save(net.state_dict(), weights_path)
        









Using device: mps
Dataloader has 2592 batches


Progress for Epoch 0:   1%|▏         | 34/2592 [00:24<31:08,  1.37it/s] 


KeyboardInterrupt: 